# 06. LightGBM — 팀원 41종 + 같은 분 같은 쌍 피처로 무패턴을 잡는가

| 모델 | 입력 | 열 |
|---|---|---:|
| `v10` | 엣지 14 (v10 17 − `amt_z_pay_ccy`, `pair_history_count`, `same_entity`) + 송신·수신 계좌 19 × 2 | 52 |
| `team41` | `v10` + 팀원 신규 8종 12열 (05) — 팀원 최종 41종·64열과 같은 구성 | 64 |
| `team41_pair` | `team41` + 같은 분 같은 쌍 6열 (05) | 70 |

- 분할: Train 8/8~10/2 · Val 10/3~10/16 · Test 10/17~10/30 (binary_classification 과 같음)
- Val: binary_classification 과 **같은 고정 검증 표본**(양성 전부 + 날짜별 정상 2%, 가중치). 조기 종료와 임계값 선택에 쓴다
- Train: 양성 전부 + 정상 `TRAIN_NEG_RATE` 표본(가중치 1/비율). 전체 1억 행 × 70열은 메모리 한도를 넘는다
- Test: **구간 전체** 2,680만 건
- 운영점: Val 에서 **P ≥ 70%** 를 지키는 가장 낮은 임계값 → 그 임계값을 Test 에 그대로 적용

무패턴 지표는 hot 15계좌(04 진단)와 나머지를 나눠 본다.

## 05 피처 두 열 보정

05 점검에서 팀원 수치와 어긋난 두 열을 여기서 바로잡는다. 나머지 10열은 단변량 AUC가 팀원과 0.003 이내로 같았다.

| 열 | 05 결과 | 팀원 | 원인(추정) | 06 처리 |
|---|---|---|---|---|
| `minute_sender_vs_prior_hour` | AUC ACH 0.632 / 비ACH 0.671 | 0.752 / 0.884 | "동일 분 비자기 송금"의 해석 | 정의 4가지를 계산해 **팀원 AUC에 가장 가까운 것**을 고른다 |
| `minute_receiver_vs_prior_hour` | 0.613 / 0.501 | 0.642 / 0.500 | 〃 | 〃 |
| `src_day_ccy_log_matching_in_amount` | 정상 > 0 비율 5.48% | 1.92% | 자기이체(src = dst, 같은 통화·금액)가 **자기 자신**을 같은 금액 입금으로 센다 | 자기 자신을 빼고 다시 센다 |

같은 분 정의 후보 (`n` 같은 분 건수, `ph` 직전 60분 `[ts−60분, ts−1분]` 건수):

| 후보 | 식 |
|---|---|
| A (05) | ln(1 + 60·(n_전체 − 1)) − ln(1 + ph_전체) — 현재 거래 자신 제외 |
| B | ln(1 + 60·n_전체) − ln(1 + ph_전체) — 자신 포함 |
| C | ln(1 + 60·n_비자기) − ln(1 + ph_비자기) — 자기이체를 빼고 셈, 자신 포함 |
| D | C 에서 현재 거래 자신 제외 |

## 0. 설정

필요한 입력:

```text
parquets/HI-Large/transactions.parquet, patterns.parquet              (01_prepare_parquet)
parquets/HI-Large/features_{train,val,test}/edge_features.parquet      (02_build_features, SPLIT 별)
parquets/HI-Large/features_{train,val,test}/node_features.parquet
parquets/HI-Large/nonpaten_vs_normal/05_features/{train,val,test}/     (05)
parquets/HI-Large/sampling/val/2022-10-03_2022-10-16_neg0.02.parquet   (없으면 같은 규칙으로 만듦)
```

출력은 `parquets/HI-Large/nonpaten_vs_normal/06_lgbm/` (데이터셋, 모델, 점수, results.json). 이미 만든 데이터셋은 재사용한다.

In [1]:
# 현재 Jupyter 커널에 패키지가 없을 때만 주석을 해제해 실행합니다.
!pip install --break-system-packages --no-cache-dir -q duckdb pandas lightgbm scikit-learn pyarrow

In [2]:
from pathlib import Path
from datetime import date, timedelta
import json, time, gc
import duckdb, numpy as np, pandas as pd, pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.metrics import average_precision_score, precision_recall_curve
from IPython.display import display

DATASET = "HI-Large"
PROJECT_ROOT = None            # 자동 탐색 실패 시 Path("/workspace/money_laundry_ai_")
THREADS, MEMORY = 8, "10GB"    # LightGBM 스레드 / DuckDB 메모리
REBUILD = False                # True: 데이터셋 새로 만들기
STRICT = True                  # 구간별 거래 수가 다르면 멈춤

HISTORY = ("2022-08-01", "2022-10-31")
SPLITS = {"train": ("2022-08-08", "2022-10-02"), "val": ("2022-10-03", "2022-10-16"), "test": ("2022-10-17", "2022-10-30")}
EXPECTED_ROWS = {"train": 102_336_629, "val": 24_413_046, "test": 26_808_230}
TRAIN_NEG_RATE = 0.10          # 학습 정상 표본 비율 (가중치 1/비율)
VAL_NEG_RATE = 0.02            # binary_classification 고정 검증 표본과 같은 값
HOT_MIN = 300                  # 04 와 같은 hot 기준 (8/1~10/30 무패턴 송신 건수)
TARGET_P = 0.70                # 운영점 정밀도
MINUTE_VARIANT = None          # None: 팀원 AUC 에 가장 가까운 후보 자동 선택, 또는 {"sender": "C", "receiver": "C"}
SEED = 42

LGB_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_child_weight=1.0, reg_lambda=1.0,
                  bagging_fraction=0.8, bagging_freq=1, feature_fraction=1.0, max_delta_step=0.7,
                  scale_pos_weight=6, metric="average_precision", seed=SEED, num_threads=THREADS, verbose=-1)
ROUNDS, EARLY_STOP = 1000, 50  # binary_classification 트리 설정(num_leaves 63, max_delta_step 0.7, spw 6)과 같고 라운드만 늘림

EDGE14 = ["pair_prior_count", "pair_gap_minutes", "pair_count_1h", "sender_gap_minutes", "sender_count_1h",
          "receiver_gap_minutes", "receiver_count_1h", "paid_currency_z", "reverse_history_count", "pair_age_at_day_start",
          "same_known_country", "both_countries_known", "same_currency", "is_self_transfer"]
NODE19 = ["log_out_count", "log_in_count", "log_out_neighbors", "log_in_neighbors", "log_self_count",
          "log_reciprocal_neighbors", "mean_paid_currency_z", "paid_outlier_fraction", "log_currency_count", "log_active_days",
          "type_Corporation", "type_Partnership", "type_Sole Proprietorship", "type_Individual", "type_Country", "type_Direct",
          "log_entity_observed_accounts", "log_entity_out_diversity", "log_entity_in_diversity"]
TEAM12 = ["minute_sender_vs_prior_hour", "minute_receiver_vs_prior_hour", "src_day_ccy_flow_balance", "dst_day_ccy_flow_balance",
          "src_day_ccy_log_matching_in_amount", "src_prior_ccy_amount_seen", "dst_prior_ccy_amount_seen",
          "src_prior_ccy_amount_vs_mean", "dst_prior_ccy_amount_vs_mean", "src_day_ccy_rate_vs_prior", "dst_day_ccy_rate_vs_prior",
          "pair_amount_seen_before_day"]
PAIR6 = ["pair_same_minute_other_n", "pair_same_minute_log_amount_ratio", "pair_same_minute_is_max", "pair_count_7d",
         "receiver_same_minute_other_in_n", "receiver_same_minute_other_in_neighbors"]
col = lambda s: s.replace(" ", "_")
NODE_COLS = [f"src_{col(n)}" for n in NODE19] + [f"dst_{col(n)}" for n in NODE19]
MODELS = {"v10": EDGE14 + NODE_COLS, "team41": EDGE14 + NODE_COLS + TEAM12, "team41_pair": EDGE14 + NODE_COLS + TEAM12 + PAIR6}
assert [len(v) for v in MODELS.values()] == [52, 64, 70]
TEAM_REF = {"HGB25(팀원)": {"val": (70.11, 63.01, 9.95), "test": (66.52, 63.23, 11.88)},
            "최종 41종(팀원)": {"val": (70.30, 62.69, 8.97), "test": (66.36, 62.72, 10.40)}}   # (P, R, 무패턴 R) % — 팀원 분할 기준


def find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "parquets" / DATASET / "transactions.parquet").is_file():
            return c
    raise FileNotFoundError("transactions.parquet 를 찾지 못했습니다. PROJECT_ROOT 를 지정하세요.")


ROOT = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else find_root()
BASE = ROOT / "parquets" / DATASET
TX, PAT = BASE / "transactions.parquet", BASE / "patterns.parquet"
F05 = BASE / "nonpaten_vs_normal" / "05_features"
OUT = BASE / "nonpaten_vs_normal" / "06_lgbm"
VAL_SAMPLE = BASE / "sampling" / "val" / f"{SPLITS['val'][0]}_{SPLITS['val'][1]}_neg{VAL_NEG_RATE:g}.parquet"
for s, (a, b) in SPLITS.items():
    fj = json.loads((BASE / f"features_{s}" / "features.json").read_text(encoding="utf-8"))
    assert str(fj.get("context_end"))[:10] == b, f"features_{s} 의 context_end={fj.get('context_end')} ≠ {b} — 02 를 SPLIT={s} 로 다시"
    assert (F05 / s).is_dir(), f"05 출력 없음: {F05 / s}"
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT); print("OUT :", OUT); print("VAL :", VAL_SAMPLE, "(있음)" if VAL_SAMPLE.exists() else "(없음 → 만듦)")

ROOT: /workspace/money_laundry_ai_
OUT : /workspace/money_laundry_ai_/parquets/HI-Large/nonpaten_vs_normal/06_lgbm
VAL : /workspace/money_laundry_ai_/parquets/HI-Large/sampling/val/2022-10-03_2022-10-16_neg0.02.parquet (있음)


In [3]:
con = duckdb.connect(str(OUT / "work.duckdb"))
con.execute(f"SET threads={THREADS}")
con.execute(f"SET memory_limit='{MEMORY}'")
con.execute(f"SET temp_directory='{OUT / 'tmp'}'")
con.execute("SET preserve_insertion_order=false")


def q(v):
    return "'" + str(v).replace("'", "''") + "'"


def has(name):
    return con.execute("SELECT count(*) FROM duckdb_tables() WHERE table_name = ?", [name]).fetchone()[0] > 0


def step(name, sql):
    if has(name) and not REBUILD:
        print(f"{name:10s} 이미 있음 — 건너뜀"); return
    t = time.perf_counter()
    con.execute(f"CREATE OR REPLACE TABLE {name} AS {sql}")
    print(f"{name:10s} {con.execute(f'SELECT count(*) FROM {name}').fetchone()[0]:>13,} 행  ({time.perf_counter() - t:,.1f}s)", flush=True)


def days(a, b):
    d0, d1 = date.fromisoformat(a), date.fromisoformat(b)
    return [str(d0 + timedelta(i)) for i in range((d1 - d0).days + 1)]

## 1. 라벨 · hot · 자기 일치 표시 `lab`

분할 기간(8/8~10/30) 거래마다: `cls` 0 정상 · 1 패턴 · 2 무패턴, `y` = 양성, `hot` = hot 15계좌 송신, `self_match` = 자기이체이면서 같은 통화·같은 금액(보정용).

In [4]:
step("lab", f"""
    WITH t AS (SELECT t.*, (CASE WHEN NOT t.is_laundering THEN 0 WHEN p.edge_id IS NOT NULL THEN 1 ELSE 2 END)::TINYINT AS cls
               FROM read_parquet({q(TX)}) t
               LEFT JOIN (SELECT DISTINCT edge_id FROM read_parquet({q(PAT)}) WHERE NOT unmatched) p ON p.edge_id = t.edge_id
               WHERE t.ts >= TIMESTAMP {q(HISTORY[0])} AND t.ts < TIMESTAMP {q(HISTORY[1])}),
         hot AS (SELECT src FROM t WHERE cls = 2 GROUP BY src HAVING count(*) >= {HOT_MIN})
    SELECT t.edge_id, t.ts, t.day, t.src, t.dst, t.cls, (t.cls > 0)::TINYINT AS y, t.payment_format AS fmt,
           (h.src IS NOT NULL) AS hot,
           (t.src = t.dst AND t.payment_currency = t.receiving_currency AND t.paid = t.received) AS self_match
    FROM t LEFT JOIN hot h ON h.src = t.src
    WHERE t.day BETWEEN DATE {q(SPLITS['train'][0])} AND DATE {q(SPLITS['test'][1])}
""")
cnt = con.execute("""SELECT CASE WHEN day <= DATE '{tr}' THEN 'train' WHEN day <= DATE '{va}' THEN 'val' ELSE 'test' END AS split,
                            count(*) AS rows, count(*) FILTER (WHERE cls = 1) AS pattern, count(*) FILTER (WHERE cls = 2) AS nonpattern,
                            count(*) FILTER (WHERE cls = 2 AND hot) AS nonpattern_hot
                     FROM lab GROUP BY 1""".format(tr=SPLITS["train"][1], va=SPLITS["val"][1])).df().set_index("split").loc[list(SPLITS)]
display(cnt)
assert cnt.rows.to_dict() == EXPECTED_ROWS or not STRICT, cnt.rows.to_dict()
print("hot 계좌:", con.execute("SELECT count(DISTINCT src) FROM lab WHERE hot").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

lab          153,557,905 행  (87.5s)


,rows,pattern,nonpattern,nonpattern_hot
split,,,,
train,102336629,62889,47782,13802
val,24413046,20100,13695,3477
test,26808230,20187,14273,3578


hot 계좌: 15


## 2. 05 보정

### 2.1 같은 분 정의 후보 계산

계좌·분마다 전체 건수와 자기이체를 뺀 건수, 각각의 직전 60분 건수. 이력은 8/1 부터.

In [5]:
for name, acct in [("mo", "src"), ("mi", "dst")]:
    step(name, f"""
        SELECT acct, ts, n_all, n_ns,
               coalesce(sum(n_all) OVER w, 0) AS ph_all, coalesce(sum(n_ns) OVER w, 0) AS ph_ns
        FROM (SELECT {acct} AS acct, ts, count(*) AS n_all, count(*) FILTER (WHERE src <> dst) AS n_ns
              FROM read_parquet({q(TX)})
              WHERE ts >= TIMESTAMP {q(HISTORY[0])} AND ts < TIMESTAMP {q(HISTORY[1])} GROUP BY ALL)
        WINDOW w AS (PARTITION BY acct ORDER BY ts RANGE BETWEEN INTERVAL 60 MINUTE PRECEDING AND INTERVAL 1 MINUTE PRECEDING)
    """)


def minute_sql(a, v, self_col="(l.src = l.dst)"):
    """후보 v 의 식. a = mo / mi 별칭"""
    ns_self = f"CASE WHEN NOT {self_col} THEN 1 ELSE 0 END"
    return {"A": f"ln(1 + 60 * ({a}.n_all - 1)) - ln(1 + {a}.ph_all)",
            "B": f"ln(1 + 60 * {a}.n_all) - ln(1 + {a}.ph_all)",
            "C": f"ln(1 + 60 * {a}.n_ns) - ln(1 + {a}.ph_ns)",
            "D": f"ln(1 + 60 * ({a}.n_ns - {ns_self})) - ln(1 + {a}.ph_ns)"}[v]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mo           148,301,973 행  (76.7s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

mi           163,975,984 행  (75.9s)


### 2.2 후보별 단변량 AUC — 팀원 10/9–13 값과 비교

05 점검과 같은 표본(무패턴 전부 + 정상 5%, `hash(edge_id)`), 방향 보정.

In [6]:
TEAM_AUC = {"sender": (0.752, 0.884), "receiver": (0.642, 0.500)}   # (ACH, 비ACH)


def auc(score, y):
    r = pd.Series(score).rank(method="average").to_numpy()
    n1 = int(y.sum()); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 and n0 else np.nan


cols = ", ".join(f"({minute_sql('o', v)})::FLOAT AS s_{v}, ({minute_sql('i', v)})::FLOAT AS r_{v}" for v in "ABCD")
cal = con.execute(f"""
    SELECT l.cls, l.fmt = 'ACH' AS ach, {cols}
    FROM lab l JOIN mo o ON o.acct = l.src AND o.ts = l.ts JOIN mi i ON i.acct = l.dst AND i.ts = l.ts
    WHERE l.day BETWEEN DATE '2022-10-09' AND DATE '2022-10-13' AND (l.cls = 2 OR (l.cls = 0 AND hash(l.edge_id) % 20 = 0))
""").df()
rows = []
for side, p in [("sender", "s"), ("receiver", "r")]:
    for v in "ABCD":
        r = {"side": side, "variant": v}
        for ach, nm in [(True, "ACH"), (False, "비ACH")]:
            d = cal[cal.ach == ach]; a = auc(d[f"{p}_{v}"].to_numpy(), (d.cls == 2).to_numpy().astype(int))
            r[nm] = round(max(a, 1 - a), 3)
        r["팀원_ACH"], r["팀원_비ACH"] = TEAM_AUC[side]
        r["차이"] = round(abs(r["ACH"] - r["팀원_ACH"]) + abs(r["비ACH"] - r["팀원_비ACH"]), 3)
        rows.append(r)
cal_tab = pd.DataFrame(rows)
display(cal_tab)
chosen = MINUTE_VARIANT or {s: cal_tab[cal_tab.side == s].sort_values("차이").variant.iloc[0] for s in ("sender", "receiver")}
print("선택:", chosen)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,side,variant,ACH,비ACH,팀원_ACH,팀원_비ACH,차이
0,sender,A,0.632,0.671,0.752,0.884,0.333
1,sender,B,0.723,0.887,0.752,0.884,0.032
2,sender,C,0.755,0.884,0.752,0.884,0.003
3,sender,D,0.641,0.681,0.752,0.884,0.314
4,receiver,A,0.613,0.501,0.642,0.500,0.030
5,receiver,B,0.579,0.502,0.642,0.500,0.065
6,receiver,C,0.643,0.500,0.642,0.500,0.001
7,receiver,D,0.594,0.503,0.642,0.500,0.051


선택: {'sender': 'C', 'receiver': 'C'}


### 2.3 같은 금액 입금 횟수에서 자기 자신 빼기

`src_day_ccy_log_matching_in_amount = ln(1 + c)` 이므로 `c = round(exp(x) − 1)` 로 되돌려 `self_match` 면 1을 뺀다. 팀원 수치(정상 1.92%, 무패턴 0.59%, 패턴 51.87%, 전수)와 비교한다.

In [7]:
MATCH_FIX = "ln(1 + greatest(round(exp(f.src_day_ccy_log_matching_in_amount) - 1) - CASE WHEN l.self_match THEN 1 ELSE 0 END, 0))"
display(con.execute(f"""
    SELECT l.cls, 100 * avg((f.src_day_ccy_log_matching_in_amount > 0)::INT) AS before_pct,
           100 * avg(({MATCH_FIX} > 0)::INT) AS after_pct
    FROM lab l JOIN read_parquet({q(F05 / 'val' / '*.parquet')}) f USING (edge_id) GROUP BY 1 ORDER BY 1
""").df().round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cls,before_pct,after_pct
0,0,1.91,1.57
1,1,51.89,51.50
2,2,0.62,0.62


## 3. 데이터셋

열: `edge_id, day, y, cls, hot, fmt, w` + 엣지 14 + 계좌 38 + 팀원 12(보정) + 같은 분 쌍 6.
계좌 피처는 `features_<split>` 의 구간 끝 프로필이다(02 와 같음, 구간 안 미래 활동 포함). 세 모델에 똑같이 들어가므로 **모델 간 비교에는 영향이 없다**.

- train: 양성 전부 + 정상 중 `hash(edge_id) % 1000 < TRAIN_NEG_RATE × 1000`
- val: 고정 검증 표본의 `edge_id`, 가중치 `w`
- test: 날짜별 전체

In [8]:
def build_sql(split, where, weight):
    e = BASE / f"features_{split}"
    node = lambda a, k: ", ".join(f'coalesce({a}."{n}", 0)::FLOAT AS "{k}_{col(n)}"' for n in NODE19)
    team = []
    for c in TEAM12:
        if c == "minute_sender_vs_prior_hour":
            team.append(f'({minute_sql("o", chosen["sender"])})::FLOAT AS "{c}"')
        elif c == "minute_receiver_vs_prior_hour":
            team.append(f'({minute_sql("i", chosen["receiver"])})::FLOAT AS "{c}"')
        elif c == "src_day_ccy_log_matching_in_amount":
            team.append(f'({MATCH_FIX})::FLOAT AS "{c}"')
        else:
            team.append(f'f."{c}"')
    return f"""
    SELECT l.edge_id, l.day, l.y, l.cls, l.hot, l.fmt, ({weight})::FLOAT AS w,
           {", ".join(f'e."{c}"::FLOAT AS "{c}"' for c in EDGE14)},
           {node("s", "src")}, {node("d", "dst")},
           {", ".join(team)},
           {", ".join(f'f."{c}"' for c in PAIR6)}
    FROM lab l
    JOIN read_parquet({q(e / 'edge_features.parquet')}) e ON e.edge_id = l.edge_id
    JOIN read_parquet({q(F05 / split / '*.parquet')}) f ON f.edge_id = l.edge_id
    JOIN mo o ON o.acct = l.src AND o.ts = l.ts
    JOIN mi i ON i.acct = l.dst AND i.ts = l.ts
    LEFT JOIN read_parquet({q(e / 'node_features.parquet')}) s ON s.node_id = l.src
    LEFT JOIN read_parquet({q(e / 'node_features.parquet')}) d ON d.node_id = l.dst
    WHERE {where}
    """


def write(sql, path):
    if path.exists() and not REBUILD:
        return pq.ParquetFile(path).metadata.num_rows
    part = path.with_suffix(".partial")
    con.execute(f"COPY ({sql}) TO {q(part)} (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000)")
    part.rename(path)
    return pq.ParquetFile(path).metadata.num_rows

In [9]:
# 고정 검증 표본 (binary_classification bc_lib.val_sample 과 같은 규칙: 날짜별 정상 2%, seed 0)
if not VAL_SAMPLE.exists():
    a, b = SPLITS["val"]
    labv = con.execute(f"SELECT edge_id, day, y FROM lab WHERE day BETWEEN DATE {q(a)} AND DATE {q(b)} ORDER BY edge_id").df()
    rng = np.random.default_rng(0); parts = [labv[labv.y == 1].assign(w=1.0)]
    for day, g in labv[labv.y == 0].groupby("day"):
        k = max(1, int(round(len(g) * VAL_NEG_RATE))); parts.append(g.iloc[rng.choice(len(g), k, replace=False)].assign(w=len(g) / k))
    VAL_SAMPLE.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(parts).sort_values("edge_id").reset_index(drop=True).to_parquet(VAL_SAMPLE, index=False)
con.execute(f"CREATE OR REPLACE TABLE val_w AS SELECT edge_id, w FROM read_parquet({q(VAL_SAMPLE)})")

t0 = time.perf_counter()
(a, b), (va, vb), (ta, tb) = SPLITS["train"], SPLITS["val"], SPLITS["test"]
k = int(round(TRAIN_NEG_RATE * 1000))
n_train = write(build_sql("train", f"l.day BETWEEN DATE {q(a)} AND DATE {q(b)} AND (l.y = 1 OR hash(l.edge_id) % 1000 < {k})",
                          f"CASE WHEN l.y = 1 THEN 1.0 ELSE {1 / TRAIN_NEG_RATE} END"), OUT / "train.parquet")
print(f"train {n_train:,} 행 ({time.perf_counter() - t0:,.0f}s)", flush=True)
n_val = write(build_sql("val", f"l.day BETWEEN DATE {q(va)} AND DATE {q(vb)} AND l.edge_id IN (SELECT edge_id FROM val_w)",
                        "(SELECT w FROM val_w v WHERE v.edge_id = l.edge_id)"), OUT / "val.parquet")
print(f"val   {n_val:,} 행 ({time.perf_counter() - t0:,.0f}s)", flush=True)
(OUT / "test").mkdir(exist_ok=True)
n_test = 0
for d in days(ta, tb):
    n_test += write(build_sql("test", f"l.day = DATE {q(d)}", "1.0"), OUT / "test" / f"{d}.parquet")
print(f"test  {n_test:,} 행 ({time.perf_counter() - t0:,.0f}s)")
assert n_test == EXPECTED_ROWS["test"] or not STRICT
assert n_val == con.execute("SELECT count(*) FROM val_w").fetchone()[0]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train 10,329,485 행 (127s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

val   521,382 행 (252s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (9 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

test  26,808,230 행 (1,133s)


In [10]:
def load(path, cols=None):
    return pq.read_table(path, columns=cols).to_pandas(strings_to_categorical=True)   # fmt 문자열 → category (메모리)


meta_cols = ["edge_id", "y", "cls", "hot", "fmt", "w"]
ALL = MODELS["team41_pair"]
tr = load(OUT / "train.parquet", meta_cols + ALL)
va = load(OUT / "val.parquet", meta_cols + ALL)
for nm, d in [("train", tr), ("val", va)]:
    print(f"{nm}: {len(d):,} 행 · 양성 {int(d.y.sum()):,} (패턴 {int((d.cls == 1).sum()):,}, 무패턴 {int((d.cls == 2).sum()):,}) · "
          f"가중 정상 {d.w[d.y == 0].sum():,.0f}")
bad = {c: int((~np.isfinite(tr[c].to_numpy())).sum()) for c in ALL}
assert sum(bad.values()) == 0, {k: v for k, v in bad.items() if v}

train: 10,329,485 행 · 양성 110,671 (패턴 62,889, 무패턴 47,782) · 가중 정상 102,188,144
val: 521,382 행 · 양성 33,795 (패턴 20,100, 무패턴 13,695) · 가중 정상 24,379,250


## 4. 학습

세 모델 모두 같은 데이터·설정. Val 가중 AP 로 조기 종료한다.

In [11]:
models, history = {}, {}
for name, feats in MODELS.items():
    t0 = time.perf_counter()
    dtr = lgb.Dataset(tr[feats].to_numpy(np.float32), label=tr.y.to_numpy(), weight=tr.w.to_numpy(), feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(va[feats].to_numpy(np.float32), label=va.y.to_numpy(), weight=va.w.to_numpy(), reference=dtr)
    ev = {}
    m = lgb.train(LGB_PARAMS, dtr, num_boost_round=ROUNDS, valid_sets=[dva], valid_names=["val"],
                  callbacks=[lgb.early_stopping(EARLY_STOP, first_metric_only=True, verbose=False), lgb.record_evaluation(ev)])
    models[name], history[name] = m, ev["val"]["average_precision"]
    m.save_model(str(OUT / f"model_{name}.txt"), num_iteration=m.best_iteration)
    print(f"{name:12s} {len(feats)}열  best_it {m.best_iteration:4d}  val AP {max(history[name]):.4f}  ({time.perf_counter() - t0:,.0f}s)", flush=True)
    del dtr, dva; gc.collect()

v10          52열  best_it  856  val AP 0.6672  (480s)
team41       64열  best_it  566  val AP 0.7032  (398s)
team41_pair  70열  best_it  618  val AP 0.7042  (441s)


## 5. 평가

- **Val**: 가중 PR 곡선에서 P ≥ 70% 인 가장 낮은 임계값(= 그 정밀도에서 최대 재현율)을 고른다
- **Test (val 임계값)**: 위 임계값을 그대로 적용 — 운영과 같은 조건
- **Test (oracle)**: test 에서 직접 P ≥ 70% 임계값을 다시 고른 참고값
- **무패턴 AP**: 패턴 양성을 빼고(정상 vs 무패턴만) 계산한 AP

In [12]:
def pick_threshold(y, s, w, target=TARGET_P):
    p, r, th = precision_recall_curve(y, s, sample_weight=w)
    ok = np.where(p[:-1] >= target)[0]
    return float(th[ok[np.argmax(r[ok])]]) if len(ok) else float("inf")


def metrics(d, s, thr):
    y, w, cls, hot = d.y.to_numpy(), d.w.to_numpy(), d.cls.to_numpy(), d.hot.to_numpy()
    a = s >= thr
    wp = (w * y).sum()
    r = {"AP": average_precision_score(y, s, sample_weight=w)}
    m = cls != 1
    r["AP_무패턴"] = average_precision_score(cls[m] == 2, s[m], sample_weight=w[m])
    r["threshold"] = thr
    r["P"] = (w * y * a).sum() / max((w * a).sum(), 1e-12)
    r["R"] = (w * y * a).sum() / wp
    r["R_패턴"] = a[cls == 1].mean() if (cls == 1).any() else np.nan
    r["R_무패턴"] = a[cls == 2].mean()
    r["R_무패턴_hot"] = a[(cls == 2) & hot].mean() if ((cls == 2) & hot).any() else np.nan
    r["R_무패턴_나머지"] = a[(cls == 2) & ~hot].mean()
    r["알림_건수"] = (w * a).sum()
    return r


val_scores = {n: m.predict(va[MODELS[n]].to_numpy(np.float32), num_iteration=m.best_iteration) for n, m in models.items()}
thr = {n: pick_threshold(va.y.to_numpy(), s, va.w.to_numpy()) for n, s in val_scores.items()}

# test: 날짜별로 읽어 점수만 모은다
test_parts = []
for f in sorted((OUT / "test").glob("*.parquet")):
    d = load(f, meta_cols + ALL)                # ALL 에 쌍둥이 확인용 두 열도 들어 있음
    part = d[meta_cols + ["pair_same_minute_other_n", "pair_count_7d"]].copy()
    for n, m in models.items():
        part[f"s_{n}"] = m.predict(d[MODELS[n]].to_numpy(np.float32), num_iteration=m.best_iteration).astype(np.float32)
    test_parts.append(part); del d; gc.collect()
te = pd.concat(test_parts, ignore_index=True); del test_parts
te[["edge_id", "cls", "hot"] + [f"s_{n}" for n in models]].to_parquet(OUT / "test_scores.parquet", index=False)
print(f"test 점수 {len(te):,} 행")

res = []
for n in models:
    res.append({"모델": n, "평가": "val", **metrics(va, val_scores[n], thr[n])})
    res.append({"모델": n, "평가": "test (val 임계값)", **metrics(te, te[f"s_{n}"].to_numpy(), thr[n])})
    t_or = pick_threshold(te.y.to_numpy(), te[f"s_{n}"].to_numpy(), te.w.to_numpy())
    res.append({"모델": n, "평가": "test (oracle)", **metrics(te, te[f"s_{n}"].to_numpy(), t_or)})
res = pd.DataFrame(res)
pct = ["P", "R", "R_패턴", "R_무패턴", "R_무패턴_hot", "R_무패턴_나머지"]
show = res.copy(); show[pct] = (100 * show[pct]).round(2); show[["AP", "AP_무패턴"]] = show[["AP", "AP_무패턴"]].round(4)
show["알림_건수"] = show["알림_건수"].round(0).astype(int)
display(show.set_index(["모델", "평가"]))
print("참고 — 팀원 (P, R, 무패턴 R) %, 팀원 분할(val 10/9–13, test 10/14–31):", TEAM_REF)

test 점수 26,808,230 행


AP  AP_무패턴  threshold          P          R  \
모델          평가                                                                
v10         val             0.6672  0.0449   0.463261  70.110001  59.689999   
            test (val 임계값)  0.6288  0.0371   0.463261  24.830000  69.570000   
            test (oracle)   0.6288  0.0371   0.850879  70.000000  55.680000   
team41      val             0.7032  0.0794   0.414623  70.089996  63.110001   
            test (val 임계값)  0.6808  0.0702   0.414623  29.410000  71.930000   
            test (oracle)   0.6808  0.0702   0.795643  70.000000  60.509998   
team41_pair val             0.7042  0.0839   0.415310  70.080002  63.060001   
            test (val 임계값)  0.6827  0.0757   0.415310  30.240000  71.190002   
            test (oracle)   0.6827  0.0757   0.800251  70.000000  60.689999   

                             R_패턴  R_무패턴  R_무패턴_hot  R_무패턴_나머지  알림_건수  
모델          평가                                                         
v10         val             97.83   3.72        0.0       4.99  28773  
            test (val 임계값)  98.58  28.54        0.0      38.09  96561  
            test (oracle)   92.64   3.41        0.0       4.55  27412  
team41      val             98.81  10.71        0.0      14.36  30428  
            test (val 임계값)  99.32  33.18        0.0      44.28  84274  
            test (oracle)   95.99  10.33        0.0      13.78  29788  
team41_pair val             98.72  10.73        0.0      14.39  30412  
            test (val 임계값)  99.10  31.70        0.0      42.31  81108  
            test (oracle)   95.95  10.81        0.0      14.43  29875

참고 — 팀원 (P, R, 무패턴 R) %, 팀원 분할(val 10/9–13, test 10/14–31): {'HGB25(팀원)': {'val': (70.11, 63.01, 9.95), 'test': (66.52, 63.23, 11.88)}, '최종 41종(팀원)': {'val': (70.3, 62.69, 8.97), 'test': (66.36, 62.72, 10.4)}}


## 6. 무패턴 세부

1. 결제방식별 무패턴 재현율 (test, val 임계값)
2. 쌍둥이 칸 — 같은 분 같은 쌍 다른 거래 1건 · 쌍의 최근 7일 거래 0건 — 에서 무패턴·패턴·정상 알림 비율

In [13]:
rows = []
for n in models:
    a = te[f"s_{n}"].to_numpy() >= thr[n]
    for fmt, g in te[te.cls == 2].groupby("fmt", observed=True):
        rows.append({"모델": n, "결제방식": fmt, "무패턴": len(g), "R_무패턴_%": round(100 * a[g.index].mean(), 2)})
pd.DataFrame(rows).pivot(index="결제방식", columns="모델", values="R_무패턴_%").join(
    te[te.cls == 2].groupby("fmt", observed=True).size().rename("무패턴_건수"))

,team41,team41_pair,v10,무패턴_건수
결제방식,,,,
ACH,44.60,42.55,38.69,10387
Bitcoin,13.01,15.04,5.69,246
Cash,0.00,0.00,0.00,523
Cheque,3.64,3.49,2.10,1949
Credit Card,0.00,0.00,0.00,1168


In [14]:
twin = (te.pair_same_minute_other_n == 1) & (te.pair_count_7d == 0)
rows = []
for n in models:
    a = te[f"s_{n}"].to_numpy() >= thr[n]
    for c, nm in [(0, "정상"), (1, "패턴"), (2, "무패턴")]:
        m = twin.to_numpy() & (te.cls.to_numpy() == c)
        rows.append({"모델": n, "라벨": nm, "건수": int(m.sum()), "알림_%": round(100 * a[m].mean(), 2) if m.any() else np.nan})
print("쌍둥이 칸 (test)")
pd.DataFrame(rows).pivot(index="라벨", columns="모델", values="알림_%").join(pd.DataFrame(rows).groupby("라벨").건수.first())

쌍둥이 칸 (test)


,team41,team41_pair,v10,건수
라벨,,,,
무패턴,98.17,97.71,13.99,1094
정상,6.46,5.35,4.39,39122
패턴,98.26,95.87,96.25,1838


## 7. 피처 중요도 (`team41_pair`, gain)

In [15]:
m = models["team41_pair"]
imp = pd.DataFrame({"feature": m.feature_name(), "gain": m.feature_importance("gain"), "split": m.feature_importance("split")})
imp["gain_%"] = (100 * imp.gain / imp.gain.sum()).round(2)
imp["묶음"] = imp.feature.map(lambda f: "팀원12" if f in TEAM12 else "같은분쌍6" if f in PAIR6 else "계좌38" if f in NODE_COLS else "엣지14")
display(imp.groupby("묶음")["gain_%"].sum().sort_values(ascending=False))
imp.sort_values("gain", ascending=False).head(25).reset_index(drop=True)

묶음
팀원12     46.84
계좌38     39.51
엣지14     12.91
같은분쌍6     0.74
Name: gain_%, dtype: float64

,feature,gain,split,gain_%,묶음
0,minute_sender_vs_prior_hour,4.213955e+06,891,18.43,팀원12
1,pair_amount_seen_before_day,4.211292e+06,176,18.42,팀원12
2,src_log_currency_count,2.674840e+06,274,11.70,계좌38
3,paid_currency_z,1.274793e+06,2568,5.58,엣지14
4,src_log_out_neighbors,1.240654e+06,1455,5.43,계좌38
5,src_log_entity_observed_accounts,1.138113e+06,1063,4.98,계좌38
6,src_day_ccy_flow_balance,1.090129e+06,644,4.77,팀원12
7,src_log_in_neighbors,6.271322e+05,891,2.74,계좌38
8,reverse_history_count,5.105269e+05,790,2.23,엣지14
9,src_log_self_count,4.076242e+05,507,1.78,계좌38


In [16]:
out = {"splits": SPLITS, "train_neg_rate": TRAIN_NEG_RATE, "val_sample": str(VAL_SAMPLE), "target_p": TARGET_P,
       "minute_variant": chosen, "minute_calibration": cal_tab.to_dict(orient="records"),
       "lgb_params": LGB_PARAMS, "best_iteration": {n: m.best_iteration for n, m in models.items()},
       "thresholds": thr, "results": res.to_dict(orient="records"), "team_reference": TEAM_REF}
(OUT / "results.json").write_text(json.dumps(out, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
con.close()
print("저장:", OUT / "results.json")

저장: /workspace/money_laundry_ai_/parquets/HI-Large/nonpaten_vs_normal/06_lgbm/results.json
